# Step 16: Final Inference Tests

Test the recommendation engine for all 4 crops using valid historical examples from the existing datasets.

In [1]:
import sys, json
from pathlib import Path
import pandas as pd
BASE_DIR = Path('..').resolve()
if str(BASE_DIR) not in sys.path: sys.path.insert(0, str(BASE_DIR))

from src.mandimitra_recommendation import get_recommendation


In [2]:
CROPS = ['rice', 'tomato', 'wheat', 'cotton']
records = []

for crop in CROPS:
    print(f'\n{crop.upper()}')
    test_path = BASE_DIR / 'data' / 'processed' / (f'maharashtra_{crop}_test.csv')
    if not test_path.exists(): continue
    df = pd.read_csv(test_path, low_memory=False).dropna(subset=['Modal Price']).head(3)
    for _, row in df.iterrows():
        features = row.to_dict()
        rec = get_recommendation(
            crop=crop,
            market=row['Market'],
            current_price=float(row['Modal Price']),
            historical_features=features,
            recent_prices=[float(row['Modal Price'])] * 5,  # mock for simplicity
            nearby_markets=[
                {'market': 'Nearby A', 'current_price': float(row['Modal Price']) + 10, 'transport_cost_per_quintal': 20}
            ],
            transport_data={'transport_cost_per_quintal': 15},
            weather_data={'temperature': 30, 'rainfall_mm': 0},
            supply_data={'market_supply_status': 'NORMAL'}
        )
        print(f"\n  Market: {rec['market']} | Price: {rec['current_price']} | Rec: {rec['recommendation']}")
        print(f"  Reason: {rec['reason']}")
        records.append({
            'crop': crop,
            'market': rec['market'],
            'date': row['Price Date'],
            'current_price': rec['current_price'],
            'forecast': rec['predicted_price'],
            'direction': rec['direction'],
            'confidence': rec['confidence'],
            'recommendation': rec['recommendation'],
            'reason': rec['reason']
        })

examples_df = pd.DataFrame(records)
(BASE_DIR / 'outputs' / 'final').mkdir(parents=True, exist_ok=True)
examples_df.to_csv(BASE_DIR / 'outputs' / 'final' / 'market_recommendation_examples.csv', index=False)



RICE

  Market: APMC Alibagh | Price: 2600.0 | Rec: SELL TODAY
  Reason: Price is expected to remain approximately stable (change of +0.0%, within the ±8.9% uncertainty band). Market conditions are stable, supporting more reliable forecasts. Nearby mandi 'Nearby A' offers a higher estimated net price of ₹2590/quintal after transport costs. Forecast is based on persistence (current price as best estimate of near-term price). Weather context: Temp: 30°C | Rainfall: 0mm Supply context: Supply: NORMAL Recommendation: Sell today to secure the current price.

  Market: APMC Alibagh | Price: 2600.0 | Rec: SELL TODAY
  Reason: Price is expected to remain approximately stable (change of +0.0%, within the ±8.9% uncertainty band). Market conditions are stable, supporting more reliable forecasts. Nearby mandi 'Nearby A' offers a higher estimated net price of ₹2590/quintal after transport costs. Forecast is based on persistence (current price as best estimate of near-term price). Weather context: 

In [3]:
# Validation tests
print('\n--- Validation Tests ---')
try:
    get_recommendation(crop='invalid_crop', market='X', current_price=100)
except ValueError as e:
    print('Invalid crop rejected:', e)

try:
    get_recommendation(crop='rice', market='X', current_price=-10)
except ValueError as e:
    print('Invalid price rejected:', e)



--- Validation Tests ---
Invalid crop rejected: Unsupported crop: 'invalid_crop'. Supported: ['rice', 'tomato', 'wheat', 'cotton']
Invalid price rejected: current_price must be a positive number, got: -10
